In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


In [13]:
def create_synthetic_data(num_classes=5, samples_per_class=5, image_shape=(28, 28, 1)):
    data = []
    labels = []

    for class_idx in range(num_classes):
        for sample_idx in range(samples_per_class):
            # Generate random images
            image = np.random.rand(*image_shape).astype(np.float32)
            data.append(image)
            labels.append(class_idx)

    return np.array(data), np.array(labels)


In [14]:
num_classes = 5
samples_per_class = 5
x_train, y_train = create_synthetic_data(num_classes, samples_per_class)


In [15]:
def create_siamese_model(input_shape):
    input = layers.Input(shape=input_shape)

    # Shared layers
    x = layers.Flatten()(input)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)

    return models.Model(inputs=input, outputs=x)


In [16]:
input_shape = (28, 28, 1)
base_network = create_siamese_model(input_shape)



In [17]:
input_a = layers.Input(shape=input_shape)
input_b = layers.Input(shape=input_shape)

In [18]:
encoded_a = base_network(input_a)
encoded_b = base_network(input_b)


In [19]:
merged = layers.Subtract()([encoded_a, encoded_b])
merged = layers.Lambda(lambda tensors: tf.abs(tensors))(merged)



In [20]:
output = layers.Dense(1, activation='sigmoid')(merged)


In [21]:
siamese_model = models.Model(inputs=[input_a, input_b], outputs=output)

In [22]:

siamese_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [23]:
def create_pairs(x, y):
    pairs = []
    labels = []

    num_classes = np.unique(y).shape[0]
    for class_idx in range(num_classes):
        class_indices = np.where(y == class_idx)[0]

        # Create positive pairs (same class)
        for i in range(len(class_indices)):
            for j in range(i + 1, len(class_indices)):
                pairs.append((x[class_indices[i]], x[class_indices[j]]))
                labels.append(1)  # Same class

        # Create negative pairs (different classes)
        for i in range(len(class_indices)):
            other_class_idx = np.random.choice(np.delete(np.arange(num_classes), class_idx))
            other_class_indices = np.where(y == other_class_idx)[0]
            other_index = np.random.choice(other_class_indices)
            pairs.append((x[class_indices[i]], x[other_index]))
            labels.append(0)  # Different classes

    return np.array(pairs), np.array(labels)

# Generate pairs and their labels
pairs, labels = create_pairs(x_train, y_train)

# Split pairs into two separate arrays for input to the model
x1 = np.array([pair[0] for pair in pairs])
x2 = np.array([pair[1] for pair in pairs])

# Step 8: Train the siamese network
siamese_model.fit([x1, x2], labels, epochs=10, batch_size=32)

# Step 9: Example of one-shot learning evaluation
def evaluate_one_shot(model, support_set, query_image):
    predictions = []
    for support_image in support_set:
        prediction = model.predict([support_image[np.newaxis, ...], query_image[np.newaxis, ...]])
        predictions.append(prediction[0][0])
    return np.array(predictions)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.4314 - loss: 0.7351
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6063 - loss: 0.6482 
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7656 - loss: 0.6008 
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8035 - loss: 0.5439 
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8513 - loss: 0.4977 
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8853 - loss: 0.4525 
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9087 - loss: 0.3863 
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9209 - loss: 0.3514 
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9432 - loss: 0.3270 
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9632 - loss: 0.2892 


In [24]:
support_set = x_train[:1]  # One support example for one-shot learning
query_image = x_train[2]  # The image to classify
similarity_scores = evaluate_one_shot(siamese_model, support_set, query_image)
print(f"Similarity Scores: {similarity_scores}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
Similarity Scores: [0.7340682]
